# TP Seance 4 — Partie 1
## Analyse de variations du nombre de copies (CNV)

**Auteur : Marwa Zidi** — Universite Paris Cite

**Duree estimee : 45 minutes**

### Objectifs
- Indexer des fichiers BAM avec samtools
- Utiliser cnv_facets.R pour detecter des CNV sur deux cas tumoraux
- Interpreter les resultats (graphiques, fichiers VCF)

### Contexte clinique
Les deux cas analyses sont des echantillons de patients atteints d'hemopathies malignes.
L'analyse CNV permet d'identifier des gains et pertes chromosomiques souvent impliques dans la progression tumorale.

---

## 1. Navigation dans l'arborescence

Avant de commencer, reperer votre position dans le systeme de fichiers et naviguer vers le dossier de travail.

- Afficher le repertoire courant avec `pwd`
- Se deplacer dans `/root/tp_seance4/`
- Lister le contenu avec `ls`
- Se positionner dans le dossier `facets/`

In [ ]:
pwd

In [ ]:
cd /root/tp_seance4/

In [ ]:
ls

In [ ]:
cd /root/tp_seance4/facets/

In [ ]:
pwd

## 2. Indexation des fichiers BAM — Case 1

FACETS necessite des fichiers BAM **indexes** (fichiers `.bai` associes).

### Etape 2.1 — Identifier les fichiers disponibles

In [ ]:
ls ./case1/bam_tumor/

In [ ]:
ls ./case1/bam_remission/

### Etape 2.2 — Decouvrir la syntaxe de samtools index

In [ ]:
! samtools index

### Etape 2.3 — Definir les chemins des fichiers BAM

In [ ]:
bam_tumor_file = "./case1/bam_tumor/ADN449-67_processed.bam"
bam_tumor_file

### Etape 2.4 — Indexer le fichier BAM tumoral

**A completer** : ecrire la commande `samtools index` pour indexer `bam_tumor_file`

In [ ]:
# A completer
! samtools index 

### Etape 2.5 — Indexer le fichier BAM de remission

**A completer** : faire de meme pour le fichier de remission `ADN452-75_processed.bam`

In [ ]:
bam_remission_file = "./case1/bam_remission/ADN452-75_processed.bam"
# A completer
! samtools index 

## 3. Decouverte de cnv_facets.R

Consulter l'aide de l'outil pour identifier les arguments necessaires.

Les arguments cles a reperer :
- `-t` : fichier BAM tumoral
- `-n` : fichier BAM normal (remission)
- `-vcf` : fichier VCF de positions SNP
- `-T` : fichier BED des regions cibles
- `-g` : version du genome de reference
- `-o` : prefixe de sortie

In [ ]:
!Rscript /usr/local/bin/cnv_facets.R -h

## 4. Preparation des chemins — Case 1

In [ ]:
# Dossiers
facets_folder = "/root/tp_seance4/facets/"
case1_folder = f"{facets_folder}case1/"
case1_results_folder = f"{case1_folder}results/"

# Fichiers BAM
bam_tumor_file = f"{case1_folder}bam_tumor/ADN449-67_processed.bam"
bam_remission_file = f"{case1_folder}bam_remission/ADN452-75_processed.bam"

# Fichiers de reference
ref_snp_file = f"{facets_folder}ref_snp/dbsnp_138.hg19.vcf.gz"
ref_bed_file = f"{facets_folder}ref_bed/panel_hg19_ref.bed"

print('Tumor BAM    :', bam_tumor_file)
print('Remission BAM:', bam_remission_file)
print('SNP ref      :', ref_snp_file)
print('BED ref      :', ref_bed_file)
print('Results dir  :', case1_results_folder)

## 5. Lancement de l'analyse — Case 1

La commande suivante lance l'analyse CNV complete :
- Pileup des SNP avec `snp-pileup`
- Segmentation et appel CNV avec l'algorithme FACETS
- Generation des graphiques et fichiers de sortie

In [ ]:
!Rscript /usr/local/bin/cnv_facets.R \
    -mq 20 \
    -bq 20 \
    -N 4 \
    -d 50 4000 \
    -T {ref_bed_file} \
    -a {ref_bed_file} \
    -g hg19 \
    -s 1234 \
    -t {bam_tumor_file} \
    -n {bam_remission_file} \
    -vcf {ref_snp_file} \
    -o {case1_results_folder}case1

### Verification des fichiers de sortie

FACETS genere plusieurs fichiers :
- `.png` : graphiques de couverture et BAF
- `.vcf` : variants CNV en format standard
- `.csv.gz` : donnees de pileup brutes
- `.rds` : objet R avec les resultats complets

In [ ]:
! ls -lh {case1_results_folder}

## 6. Analyse du Case 2 — Echantillon non apparie

Le case2 ne dispose pas d'echantillon normal propre. On utilise l'option `-u` (unmatched)
avec le fichier de remission du case1 comme reference.

**A completer** : definir les chemins et lancer l'analyse

- `case2_folder`
- `case2_results_folder`
- `bam_tumor_file_2` (fichier : `424-96_processed.bam`)
- Indexer le BAM tumoral du case2
- Lancer cnv_facets.R avec l'option `-u`

In [ ]:
# A completer
case2_folder = # chemin a completer
case2_results_folder = # chemin a completer
bam_tumor_file_2 = # chemin a completer

print('Case2 tumor BAM:', bam_tumor_file_2)

In [ ]:
# Indexer le BAM tumor case2
# A completer
! samtools index 

In [ ]:
# A completer — ajouter l'option -u pour echantillon non apparie
!Rscript /usr/local/bin/cnv_facets.R \
    -mq 20 \
    -bq 20 \
    -N 4 \
    -d 50 4000 \
    -T {ref_bed_file} \
    -a {ref_bed_file} \
    -g hg19 \
    -s 1234 \
    # ajouter l'option unmatched ici
    -t {bam_tumor_file_2} \
    -n {bam_remission_file} \
    -vcf {ref_snp_file} \
    -o {case2_results_folder}case2

In [ ]:
! ls -lh {case2_results_folder}

## Points cles

- FACETS compare tumeur et normal pour identifier les segments CNV
- Le fichier `.png` montre le log ratio (couverture) et la BAF (balance allelique)
- L'option `-u` est utilisee quand l'echantillon normal n'est pas apparie
- Les CNV sont exportes au format VCF standard

---

**Passer au notebook suivant : `filt3r_exercice.ipynb`**